### Dependencies

In [ ]:
!pip -q install -U datasets tokenizers accelerate tqdm numpy einops imageio pillow transformers
!pip install hf_transfer

In [ ]:
vocab_size = 26_000
train_examples = 1000_000
val_examples = 10_000
tokenizer_train_examples = 150_000
train_steps = 20_000
batch_size = 32
grad_accumulation = 2
learning_rate = 2e-4
weight_decay = 0.1
warmup_steps = 1_000

In [ ]:
import os , math, time , json, random
import numpy as np
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as f
from torch.utils.data import IterableDataset, DataLoader

from datasets import load_dataset

train_ds = load_dataset("roneneldan/TinyStories", split = f"train[:{train_examples}]")
val_ds = load_dataset("roneneldan/TinyStories", split = f"validation[:{val_examples}]")
print(train_ds,val_ds)

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.normalizers import NKFC
from tokenizers.processors import TemplateProcessing

special_tokens = [
    "[PAD]" , "[UNK]" , ["BOS"] , ["EOS"] , "[MASK]",
    "<|user|>" , "<|assistant|>" , "<|system|>" , "<|end|>"
]

def tokenizer_training_iterator(ds,n_examples):
    for i in range(min(n_examples,len(ds))):
        story = ds[i]["text"].strip()
        yield f"<|user|>\nWrite a short story.\n<|assistant|>\n{story}\n<|end|>\n"

tokenizer = Tokenizer(BPE(unk_token=["UNK"]))
tokenizer.normalizer = NKFC()
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space= False)


trainer = BpeTrainer(
    vocab_size= 26_000,
    min_frequency=2,
    special_tokens= special_tokens
)

print("Training Tokenizer....")
tokenizer.train_from_iterator(
    tokenizer_training_iterator(train_ds,tokenizer_train_examples),
    trainer = trainer
)

bos_id = tokenizer.token_to_id("[BOS]")
eos_id = tokenizer.token_to_id("[EOS]")

tokenizer.post_processor = TemplateProcessing(
    single = "[BOS] $A [EOS]",
    special_tokens= [("[BOS]",bos_id),("[EOS]",eos_id)],
)

tokenizer.decoder = ByteLevelDecoder()

tokenizer_dir = "tokenizer_from_scratch"
os.makedirs(tokenizer_dir,exist_ok=True)
tokenizer_file = os.path.join(tokenizer_dir,"tokenizer.json")
tokenizer.save(tokenizer_file)

print(f"Saved Tokenizer: f{tokenizer_file}")
print(f"Vocab Size: {tokenizer.get_vocab_size()}")

In [ ]:
from transformers import PreTrainedTokenizerFast

hf_tokenizer = PreTrainedTokenizerFast(tokenizer_file=tokenizer_file)

hf_tokenizer.pad_token  = "[PAD]"
hf_tokenizer.unk_token  = "[UNK]"
hf_tokenizer.bos_token  = "[BOS]"
hf_tokenizer.eos_token  = "[EOS]"
hf_tokenizer.mask_token = "[MASK]"

hf_tokenizer.add_special_tokens({
    "additional_special_tokens": ["<|user|>", "<|assistant|>", "<|system|>", "<|end|>"]
})

PAD_ID  = hf_tokenizer.pad_token_id
MASK_ID = hf_tokenizer.mask_token_id
BOS_ID  = hf_tokenizer.bos_token_id
EOS_ID  = hf_tokenizer.eos_token_id

print("PAD_ID:", PAD_ID, "MASK_ID:", MASK_ID, "BOS_ID:", BOS_ID, "EOS_ID:", EOS_ID)
print("Example encoding:", hf_tokenizer.encode("Hello world!")[:20])

In [ ]:
from dataclasses import dataclass

@dataclass
class DiffusionLLMConfig:
    vocab_size: int 
    seq_len: int 
    d_model: int 
    n_layers: int
    n_heads: int
    d_ff: int 
    dropout: float
    diffusion_steps: int

class DiffusionTransformerLM(nn.Module):
    def __init__(self, cfg:DiffusionLLMConfig):
        super().__init__()
        self.cfg = cfg

        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.seq_len, cfg.d_model)
        self.time_emb = nn.Embedding(cfg.diffusion_steps + 1 ,cfg.d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model = cfg.d_model,
            nhead = cfg.n_heads,
            dim_feedforward= cfg.d_ff,
            dropout= cfg.dropout,
            batch_first= cfg.dropout,
            batch_first = True,
            activation= "gelu",
            norm_first= True,
        )

        self.encoder = nn.TransformerEncoder(encoder_layer=enc_layer,num_layers=cfg.n_layers)
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model,cfg.vocab_size, bias =False)

        self.lm_head.weight = self.tok_emb.weight
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self,input_ids,timesteps,attention_mask = None):
        B,L = input_ids.shape
        if L>self.cfg.seq_len:
            raise ValueError(f"Sequence Length {L} > Configuration Sequence Length {self.cfg.seq_len}")
        
        pos = torch.arange(0,L,device = input_ids.device).unsqueeze(0)
        x = self.tok_emb(input_ids) + self.pos_emb(pos)

        t_emb = self.time_emb(timesteps).unsqueeze(1)
        x = x + t_emb
        x = self.drop(x)

        if attention_mask is None:
            src_key_padding_mask = None
        else:
            src_key_padding_mask = ~attention_mask
        
        x = self.encoder(x,src_key_padding_mask = src_key_padding_mask)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits
    
model = DiffusionLLMConfig(
    vocab_size= len(hf_tokenizer),
    seq_len= 256,
    d_model = 512,
    n_layers= 10,
    n_heads = 8,
    d_ff = 4 * 512,
    dropout = 0.1,
    diffusion_steps = 128,
)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model Parameters: {n_params}")